# Sorting and index decisions · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 07. Sorting and index decisions

Order records deliberately and control which labels identify rows.

Before you start: Know filtering, loc, iloc, and table shape.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html#the-index-object

### M07C01. Sort rows by a value

sort_values() orders whole rows by one or more column values. ascending=False puts larger values first. It returns a new table, so assign the result if you want to keep the order. The index labels travel with their rows and are not automatically renumbered.

**Read:** What does ascending=False request?

Answer: Descending order, from larger scores to smaller scores.

**Predict:** Who are the top three students by Score?

Answer: Fatima with 95, Chloe with 91, and Jamal with 89.

**Explain:** Why are the printed index labels 5, 2, and 9 instead of 0, 1, and 2?

Answer: The original row labels stay attached to the records when the rows are sorted.



In [ ]:
# Predict before running: Who are the top three students by Score?
df = students()
result = df.sort_values("Score", ascending=False).head(3)
print(result[["Student_ID", "Name", "Score"]])
print("Index labels:", result.index.tolist())

**Your turn:** Find the three lowest scores instead. Keep Student_ID, Name, and Score in result.

For the check, keep these result names: `result`.

**Hint:** Sort ascending and then take head(3).

**Watch for this:** head(3) before sorting chooses the first three source rows, not the three highest scores.

In [ ]:
# Change this code to complete the task.
# Question: Find the three lowest scores instead. Keep Student_ID, Name, and Score in result.
df = students()
result = df.sort_values("Score", ascending=True).head(3)[["Student_ID", "Name", "Score"]]
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Hassan', 'Dana', 'Khalid']
assert result['Score'].tolist() == [58, 65, 68]
print("Code checks passed. Review the written explanation as well.")

The three lowest scores are Hassan 58, Dana 65, and Khalid 68. Sorting whole rows keeps each score matched with its student.

**Quick check:** Does sort_values rearrange only the selected column's values?

1. Yes, other columns stay fixed.
2. No, it reorders whole rows.
3. It always removes the column.

Answer: 2. Whole rows move together so their fields stay associated.

**Explain what you learned:** I can sort a table and identify the top or bottom records.

### M07C02. Specify a tie rule

Rows can tie on a sort column. With one column, kind="stable" preserves their existing order within a tie. To define a new tie rule, sort by several columns and give one ascending flag per column. Clear tie rules make results easier to reproduce and explain.

**Read:** Which rows share a score?

Answer: Chloe, Amira, and Dana all have 80.

**Predict:** What is their order after this stable sort?

Answer: Chloe, Amira, then Dana, preserving their order in the original table.

**Explain:** Why might a report need a second sort column?

Answer: It provides a clear order among equal values, such as alphabetical order among students with the same score.



In [ ]:
# Predict before running: What is their order after this stable sort?
df = pd.DataFrame({
    "Name": ["Chloe", "Amira", "Bilal", "Dana"],
    "Score": [80, 80, 95, 80]
})
result = df.sort_values("Score", ascending=False, kind="stable")
print(result)

**Your turn:** Sort by Score descending, then Name ascending to break ties alphabetically.

For the check, keep these result names: `result`.

**Hint:** Pass ["Score", "Name"] and ascending=[False, True].

**Watch for this:** Alphabetical tie order must be requested explicitly; a stable sort preserves source order instead.

In [ ]:
# Change this code to complete the task.
# Question: Sort by Score descending, then Name ascending to break ties alphabetically.
df = pd.DataFrame({
    "Name": ["Chloe", "Amira", "Bilal", "Dana"],
    "Score": [80, 80, 95, 80]
})
result = df.sort_values(["Score", "Name"], ascending=[False, True])
print(result)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Name'].tolist() == ['Bilal', 'Amira', 'Chloe', 'Dana']
print("Code checks passed. Review the written explanation as well.")

Bilal remains first with 95. The three scores of 80 are then ordered Amira, Chloe, Dana.

**Quick check:** What does a stable sort preserve for equal sort values?

1. The existing relative order.
2. Only the highest value.
3. Alphabetical order automatically.

Answer: 1. Stability preserves the source order of rows that tie on the sort key.

**Explain what you learned:** I can make the order of tied records explicit.

### M07C03. Set, check, and reset an index

An index labels rows; it is not automatically a unique ID. set_index() chooses the labels, and index.is_unique tells you whether any label repeats. reset_index() moves the current index back into a column and supplies new default row labels. Use reset_index(drop=True) only when you deliberately want to discard the current labels.

**Read:** What does indexed.index.is_unique tell you?

Answer: It is True when no index label repeats. Separate checks are still needed for missing or invalid IDs.

**Predict:** After reset_index(), is Student_ID a data column again?

Answer: Yes. It is restored as a column, while the row index becomes 0 through 11.

**Explain:** Why could reset_index(drop=True) lose useful information here?

Answer: Student_ID is currently the index, so drop=True would discard those IDs instead of restoring them to a column.



In [ ]:
# Predict before running: After reset_index(), is Student_ID a data column again?
df = students()
indexed = df.set_index("Student_ID")
print("Unique index:", indexed.index.is_unique)
result = indexed.reset_index()
print(result[["Student_ID", "Name"]].head(3))

**Your turn:** Set Student_ID as the index, select S003 and S001 in that order, and reset the selected index without losing the IDs.

For the check, keep these result names: `result`.

**Hint:** Use .loc[["S003", "S001"]] followed by .reset_index().

**Watch for this:** Having an index does not guarantee that its labels are unique or non-missing.

In [ ]:
# Change this code to complete the task.
# Question: Set Student_ID as the index, select S003 and S001 in that order, and reset the selected index without losing the IDs.
df = students()
indexed = df.set_index("Student_ID")
selected = indexed.loc[["S003", "S001"]]
result = selected.reset_index()
print(result[["Student_ID", "Name"]])

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result['Student_ID'].tolist() == ['S003', 'S001']
assert result.index.tolist() == [0, 1]
print("Code checks passed. Review the written explanation as well.")

The selected rows are Chloe and Amira. Their IDs remain in the Student_ID column, and their new row labels are 0 and 1.

**Quick check:** Which operation keeps the old index as a column?

1. reset_index(drop=True)
2. reset_index()
3. sort_index()

Answer: 2. reset_index() preserves the old index values in a column unless drop=True is specified.

**Explain what you learned:** I can change an index without accidentally discarding record IDs.

### M07C04. Reindex to a requested order

reindex() aligns a table to a specified set of labels and their requested order. Existing labels bring their rows with them; new labels create missing values. This differs from sorting because you supply the desired labels explicitly. Use isna() to inspect newly missing cells.

**Read:** Where did the S999 row come from?

Answer: It was requested in reindex but was absent from the source index, so pandas created a row of missing values.

**Predict:** Which order will the index have?

Answer: S003, S001, S999, matching the order passed to reindex.

**Explain:** Why is a missing S999 score not automatically a failing score?

Answer: There is no source record for S999. Missing data does not establish a measured score or a fail decision.



In [ ]:
# Predict before running: Which order will the index have?
df = students().set_index("Student_ID")
result = df.reindex(["S003", "S001", "S999"])[["Name", "Score"]]
print(result)
print("Missing name:")
print(result["Name"].isna())

**Your turn:** Reindex to S002, S001, and S004, in that order. Confirm that none of their names is missing.

For the check, keep these result names: `result`.

**Hint:** Use only existing IDs and count missing names with .isna().sum().

**Watch for this:** Reindexing can fail when source labels are duplicated; check uniqueness before using labels as identifiers.

In [ ]:
# Change this code to complete the task.
# Question: Reindex to S002, S001, and S004, in that order. Confirm that none of their names is missing.
df = students().set_index("Student_ID")
result = df.reindex(["S002", "S001", "S004"])[["Name", "Score"]]
print(result)
print("Missing names:", result["Name"].isna().sum())

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ['S002', 'S001', 'S004']
assert result['Name'].isna().sum() == 0
print("Code checks passed. Review the written explanation as well.")

The output is Bilal, Amira, Dana in the requested order, with no missing names. All requested labels exist.

**Quick check:** What can reindex do when a requested label does not exist?

1. Guess the nearest record.
2. Raise an error in every case.
3. Create a row with missing values.

Answer: 3. For this unique index, reindex creates missing data for an absent requested label.

**Explain what you learned:** I can distinguish sorting existing rows from aligning to a requested index.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates